# FinFlow 1.0 — Phase 2: FinBERT Sentiment Inference (Colab Edition)

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

This notebook scores your aligned datasets with ProsusAI/finbert and produces:
- `train_sentiment.csv`
- `test_sentiment.csv`

These files are identical to the local `phase2_finbert.py` output — plug them straight into Phase 3.

In [ ]:
!pip install transformers torch tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output folder in Drive
import os
os.makedirs('/content/drive/MyDrive/FinFlow/data/processed', exist_ok=True)
print('Drive mounted. Outputs will save to: /content/drive/MyDrive/FinFlow/data/processed/')

In [ ]:
# Upload train_aligned.csv and test_aligned.csv from: FinFlow/data/aligned/
from google.colab import files
print('Upload your two files: train_aligned.csv and test_aligned.csv')
uploaded = files.upload()
for fname in uploaded:
    print(f'  ✓ Uploaded: {fname}  ({len(uploaded[fname])/1024/1024:.1f} MB)')

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from tqdm.notebook import tqdm
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    BATCH_SIZE = 32
else:
    print('WARNING: No GPU found! Go to Runtime → Change runtime type → T4 GPU')
    BATCH_SIZE = 8

print('\nLoading ProsusAI/finbert (~440 MB)...')
MODEL_NAME = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()
print(f'Model loaded. Labels: {model.config.id2label}')
print(f'Batch size: {BATCH_SIZE}')

In [ ]:
MAX_LENGTH = 128

@torch.no_grad()
def run_batch(texts):
    """Returns np.ndarray (N, 3): [positive_prob, negative_prob, neutral_prob]"""
    encoded = tokenizer(texts, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors='pt')
    encoded = {k: v.to(device) for k, v in encoded.items()}
    logits = model(**encoded).logits
    probs = softmax(logits, dim=-1).cpu().numpy()
    return probs  # [positive, negative, neutral]

def infer_sentiment(df, split_name):
    df = df.copy()
    has_news = df['news_count'] > 0
    active_df = df[has_news].copy()
    n_active = len(active_df)
    n_silent = len(df) - n_active
    n_batches = (n_active + BATCH_SIZE - 1) // BATCH_SIZE

    print(f'\n{split_name}: {n_active:,} rows to score | {n_silent:,} neutral bypass | {n_batches} batches')

    # Neutral bypass
    df.loc[~has_news, 'sentiment_score']    = 0.0
    df.loc[~has_news, 'sentiment_positive'] = 0.0
    df.loc[~has_news, 'sentiment_negative'] = 0.0
    df.loc[~has_news, 'sentiment_neutral']  = 1.0

    indices = active_df.index.tolist()
    texts   = active_df['finbert_input'].fillna('').astype(str).tolist()

    start = time.time()
    for i in tqdm(range(n_batches), desc=f'FinBERT [{split_name}]'):
        i_s = i * BATCH_SIZE
        i_e = min(i_s + BATCH_SIZE, n_active)
        probs = run_batch(texts[i_s:i_e])  # (N, 3)
        for idx, (pos, neg, neu) in zip(indices[i_s:i_e], probs):
            df.at[idx, 'sentiment_score']    = float(pos - neg)
            df.at[idx, 'sentiment_positive'] = float(pos)
            df.at[idx, 'sentiment_negative'] = float(neg)
            df.at[idx, 'sentiment_neutral']  = float(neu)

    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min | {n_active/elapsed:.0f} texts/sec')
    return df

print('Functions defined. Ready to run.')

In [ ]:
train = pd.read_csv('train_aligned.csv', parse_dates=['timestamp'])
print(f'Train loaded: {train.shape}')

train_out = infer_sentiment(train, 'TRAIN')

# Quick sanity check
scored = train_out[train_out['news_count'] > 0]
print(f'\nTrain sentiment stats:')
print(f'  Score range : [{scored["sentiment_score"].min():.3f}, {scored["sentiment_score"].max():.3f}]')
print(f'  Mean score  : {scored["sentiment_score"].mean():.4f}')
print(f'  Bullish     : {(scored["sentiment_score"]>0.1).mean():.1%}')
print(f'  Bearish     : {(scored["sentiment_score"]<-0.1).mean():.1%}')

In [ ]:
test = pd.read_csv('test_aligned.csv', parse_dates=['timestamp'])
print(f'Test loaded: {test.shape}')

test_out = infer_sentiment(test, 'TEST')

scored_t = test_out[test_out['news_count'] > 0]
print(f'\nTest sentiment stats:')
print(f'  Score range : [{scored_t["sentiment_score"].min():.3f}, {scored_t["sentiment_score"].max():.3f}]')
print(f'  Mean score  : {scored_t["sentiment_score"].mean():.4f}')

In [ ]:
# Save to Google Drive
train_path = '/content/drive/MyDrive/FinFlow/data/processed/train_sentiment.csv'
test_path  = '/content/drive/MyDrive/FinFlow/data/processed/test_sentiment.csv'

train_out.to_csv(train_path, index=False)
test_out.to_csv(test_path,   index=False)

print(f'Saved:')
print(f'  {train_path}')
print(f'  {test_path}')
print()
print('Download these two files and place them in:')
print('  FinFlow/data/processed/train_sentiment.csv')
print('  FinFlow/data/processed/test_sentiment.csv')
print()
print('Phase 3 next: LSTM / Prophet / Transformer baseline models.')

In [ ]:
# (OPTIONAL): Per-ticker sentiment breakdown + quick chart 
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#0a0a0f')
fig.suptitle('FinBERT Sentiment Score Distribution by Ticker — Train Set',
             color='#c8cdd6', fontsize=13, fontweight='bold')

COLORS = {'NVDA': '#3b82f6', 'TSLA': '#22c55e', 'JPM': '#f59e0b', 'SPY': '#f97316'}
tickers = ['NVDA', 'TSLA', 'JPM', 'SPY']

for ax, sym in zip(axes.flat, tickers):
    color = COLORS[sym]
    sym_data = train_out[(train_out['symbol'] == sym) & (train_out['news_count'] > 0)]
    scores = sym_data['sentiment_score'].dropna()

    ax.set_facecolor('#0f1117')
    ax.hist(scores, bins=50, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(0, color='white', linewidth=1, alpha=0.5)
    ax.axvline(scores.mean(), color='red', linewidth=1.5, linestyle='--',
               label=f'Mean: {scores.mean():.3f}')

    ax.set_title(sym, color=color, fontweight='bold', fontsize=12)
    ax.set_xlabel('Sentiment Score', color='#c8cdd6')
    ax.tick_params(colors='#5a6070')
    for spine in ax.spines.values(): spine.set_edgecolor('#1e2330')
    ax.legend(facecolor='#0f1117', edgecolor='#1e2330', labelcolor='#c8cdd6', fontsize=9)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FinFlow/sentiment_dist.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('Chart saved to Drive.')